<a href="https://colab.research.google.com/github/norewyx0205/vlm-event-boundary/blob/main/notebooks/colab_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ladder Event Boundary Evaluation on Colab

This notebook keeps the baseline sanity-check evaluation and runs the 6-level ladder experiment with Qwen3-VL.


In [13]:
%cd /content
!ls

/content
sample_data  vlm-event-boundary


In [14]:
from google.colab import userdata
import os

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["GH_TOKEN"] = userdata.get("GH_TOKEN")

## Clone or Update Repository

If the repository already exists in Colab, this cell pulls the latest code. If it does not exist, it clones the repo.


In [15]:
from getpass import getpass
import os

REPO_URL = "github.com/norewyx0205/vlm-event-boundary.git"
REPO_DIR = "/content/vlm-event-boundary"

if not os.path.exists(REPO_DIR):
    token = os.environ["GH_TOKEN"]
    if token:
        !git clone https://{token}@{REPO_URL} {REPO_DIR}
    else:
        !git clone https://{REPO_URL} {REPO_DIR}
else:
    print("Repository already exists; pulling latest changes...")
    %cd {REPO_DIR}
    !git pull


Repository already exists; pulling latest changes...
/content/vlm-event-boundary
remote: Enumerating objects: 19, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 12 (delta 7), reused 9 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (12/12), 28.69 KiB | 139.00 KiB/s, done.
From https://github.com/norewyx0205/vlm-event-boundary
   5304a77..ce72fe4  main       -> origin/main
Updating 5304a77..ce72fe4
Fast-forward
 README.md                  |  11 +++
 notebooks/colab_eval.ipynb | 228 +++++++++++++++++++++++++++------------------
 scripts/analyze_results.py | 119 +++++++++++++++++++++++
 scripts/run_eval.py        |   1 +
 4 files changed, 266 insertions(+), 93 deletions(-)


In [16]:
%cd /content/vlm-event-boundary
!ls

/content/vlm-event-boundary
analysis			notebooks    scripts
baseline_boundary_videos	README.md    synthetic_boundary_videos
data				results
generate_2d_boundary_videos.py	run_eval.py


## Install Dependencies

These packages are needed for Qwen video input, video generation, and result analysis.


In [17]:
!pip install -U transformers accelerate qwen-vl-utils decord opencv-python imageio-ffmpeg

In [6]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    torch.set_default_device("cuda")


CUDA available: True


## Configuration

Qwen3-VL is the default model for the ladder experiment. You can change the model string here if needed.


In [7]:
MODEL_NAME = "Qwen/Qwen3-VL-8B-Instruct"
RESULT_DIR = "/content/vlm-event-boundary/results"

BASELINE_ANNOTATION = "/content/vlm-event-boundary/baseline_boundary_videos/annotations.jsonl"
SYNTHETIC_ANNOTATION = "/content/vlm-event-boundary/synthetic_boundary_videos/annotations.jsonl"
LADDER_ROOT = "/content/vlm-event-boundary/data/ladder_v2"
DATASET_VERSION = "ladder_v2"

## Generate Baseline and Synthetic Reference Datasets

This regenerates the legacy simple baseline and the harder synthetic reference set. These are kept as reference points outside the 6-level ladder.

In [9]:
!python generate_2d_boundary_videos.py --dataset all

Generated 120 hard videos.
Generated 240 hard evaluation rows.
Videos saved to: synthetic_boundary_videos/videos
Annotations saved to: synthetic_boundary_videos/annotations.jsonl

Hard condition video counts:
- low_boundary: 30
- temporal_boundary: 30
- visual_boundary: 30
- audio_boundary: 30

Generated 20 baseline videos.
Generated 40 baseline evaluation rows.
Videos saved to: baseline_boundary_videos/videos
Annotations saved to: baseline_boundary_videos/annotations.jsonl

Baseline condition video counts:
- low_boundary: 5
- temporal_boundary: 5
- visual_boundary: 5
- audio_boundary: 5


## Baseline Sanity Check

This keeps the earlier simple baseline. It should verify that Qwen3 can solve the easy before/after task.


In [ ]:
from pathlib import Path

for name, annotation in [
    ("baseline", BASELINE_ANNOTATION),
    ("synthetic", SYNTHETIC_ANNOTATION),
]:
    path = Path(annotation)
    print(f"{name} annotation exists:", path.exists())
    if path.exists():
        print(f"{name} eval rows:", sum(1 for _ in open(path)))
        print(f"{name} videos:", len(list((path.parent / "videos").glob("*.mp4"))))
    else:
        print(f"{name} files not found; run the generation cell above.")

In [ ]:
!python scripts/run_eval.py \
  --annotation_path "$BASELINE_ANNOTATION" \
  --model_name "$MODEL_NAME" \
  --dataset_name baseline_qwen3_sanity_check \
  --output_dir "$RESULT_DIR"

## Synthetic Hard Reference Evaluation

This evaluates the legacy harder synthetic set so it can be compared with the simple baseline and the ladder levels.

In [ ]:
!python scripts/run_eval.py \
  --annotation_path "$SYNTHETIC_ANNOTATION" \
  --model_name "$MODEL_NAME" \
  --dataset_name synthetic_qwen3_reference \
  --output_dir "$RESULT_DIR"

## Generate 6-Level Ladder Dataset

This creates the 6-level `data/ladder_v2/level_*` dataset with evaluation-level mirrored annotations. Re-run this cell when generation parameters change.


In [8]:
!python scripts/generate_ladder_dataset.py \
  --dataset_version "$DATASET_VERSION" \
  --samples_per_level 30 \
  --output_root "$LADDER_ROOT" \
  --seed 42


level_1_simple: wrote 240 eval rows to /content/vlm-event-boundary/data/ladder_v2/level_1_simple/annotations.jsonl
level_2_randomized: wrote 240 eval rows to /content/vlm-event-boundary/data/ladder_v2/level_2_randomized/annotations.jsonl
level_3_non_target_static_distractors: wrote 240 eval rows to /content/vlm-event-boundary/data/ladder_v2/level_3_non_target_static_distractors/annotations.jsonl
level_4_target_like_static_distractors: wrote 240 eval rows to /content/vlm-event-boundary/data/ladder_v2/level_4_target_like_static_distractors/annotations.jsonl
level_5_target_like_moving_distractors: wrote 240 eval rows to /content/vlm-event-boundary/data/ladder_v2/level_5_target_like_moving_distractors/annotations.jsonl
level_6_hard_temporal_interference: wrote 240 eval rows to /content/vlm-event-boundary/data/ladder_v2/level_6_hard_temporal_interference/annotations.jsonl


## Check Ladder Dataset

Each level should contain 30 base samples × 4 boundary conditions = 120 unique annotated videos, and 120 × 2 mirrored prompts = 240 evaluation rows. The full ladder has 6 levels. The check below counts videos referenced by annotations, and separately reports stale/extra `.mp4` files if the directory contains leftovers from earlier runs.


In [17]:
import json
import subprocess
from pathlib import Path

for ann in sorted(Path(LADDER_ROOT).glob("level_*/annotations.jsonl")):
    rows = [json.loads(line) for line in ann.read_text().splitlines()]
    annotated_videos = {row["video_id"] for row in rows if row.get("prompt_variant") == "original"}
    disk_videos = {path.name for path in (ann.parent / "videos").glob("*.mp4")}
    extra_mp4 = sorted(disk_videos - annotated_videos)
    missing_mp4 = sorted(annotated_videos - disk_videos)
    print(
        ann.parent.name,
        "annotated_videos=", len(annotated_videos),
        "disk_mp4=", len(disk_videos),
        "eval_rows=", len(rows),
        "extra_mp4=", len(extra_mp4),
        "missing_mp4=", len(missing_mp4),
    )
    if extra_mp4:
        print("  extra examples:", extra_mp4[:5])
    if missing_mp4:
        print("  missing examples:", missing_mp4[:5])

subprocess.run(["python", "scripts/check_ladder_dataset.py", "--root", LADDER_ROOT], check=True)


level_1_simple videos= 120 eval_rows= 240
level_2_randomized videos= 120 eval_rows= 240
level_3_non_target_static_distractors videos= 120 eval_rows= 240
level_4_target_like_static_distractors videos= 120 eval_rows= 240
level_5_target_like_moving_distractors videos= 120 eval_rows= 240
level_6_hard_temporal_interference videos= 120 eval_rows= 240


## Qwen3 Ladder Smoke Test

Run a tiny subset before launching the full ladder evaluation.


In [9]:
!python scripts/run_eval.py \
  --annotation_path "$LADDER_ROOT/level_1_simple/annotations.jsonl" \
  --model_name "$MODEL_NAME" \
  --dataset_name smoke_ladder_v2_level_1_simple_qwen3 \
  --output_dir "$RESULT_DIR" \
  --max_samples 4

config.json: 100% 1.47k/1.47k [00:00<00:00, 819kB/s]
model.safetensors.index.json: 100% 67.8k/67.8k [00:00<00:00, 121MB/s]
Fetching 4 files: 100% 4/4 [00:42<00:00, 10.56s/it]
Download complete: 100% 17.5G/17.5G [00:42<00:00, 414MB/s]
Loading weights: 100% 750/750 [00:05<00:00, 136.00it/s]
generation_config.json: 100% 269/269 [00:00<00:00, 1.28MB/s]
preprocessor_config.json: 100% 390/390 [00:00<00:00, 1.69MB/s]
chat_template.json: 100% 5.50k/5.50k [00:00<00:00, 4.07MB/s]
tokenizer_config.json: 100% 10.9k/10.9k [00:00<00:00, 23.7MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 2.99MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 8.77MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 74.3MB/s]
video_preprocessor_config.json: 100% 385/385 [00:00<00:00, 1.76MB/s]
Processing level_1_sample_001_low_boundary_original
qwen-vl-utils using torchcodec to read video.
level_1_sample_001_low_boundary.mp4 pred= A correct= A is_correct= True raw= 'A'
Processing level_1_sample_001_low_boundary_swapped

## Run Qwen3 on All 6 Ladder Levels

This is the main ladder experiment. Results are saved under `results/<safe_model_name>/<dataset_name>/<timestamp>/`.


In [10]:
LEVELS = [
    "level_1_simple",
    "level_2_randomized",
    "level_3_non_target_static_distractors",
    "level_4_target_like_static_distractors",
    "level_5_target_like_moving_distractors",
    "level_6_hard_temporal_interference",
]

for level_name in LEVELS:
    annotation_path = f"{LADDER_ROOT}/{level_name}/annotations.jsonl"
    dataset_name = f"{DATASET_VERSION}_{level_name}"
    print("Running", dataset_name)
    !python scripts/run_eval.py \
      --annotation_path "$annotation_path" \
      --model_name "$MODEL_NAME" \
      --dataset_name "$dataset_name" \
      --output_dir "$RESULT_DIR"

Running ladder_v2_level_1_simple
Loading weights: 100% 750/750 [00:05<00:00, 135.43it/s]
Processing level_1_sample_001_low_boundary_original
qwen-vl-utils using torchcodec to read video.
level_1_sample_001_low_boundary.mp4 pred= A correct= A is_correct= True raw= 'A'
Processing level_1_sample_001_low_boundary_swapped
level_1_sample_001_low_boundary.mp4 pred= B correct= B is_correct= True raw= 'B'
Processing level_1_sample_001_temporal_boundary_original
level_1_sample_001_temporal_boundary.mp4 pred= A correct= A is_correct= True raw= 'A'
Processing level_1_sample_001_temporal_boundary_swapped
level_1_sample_001_temporal_boundary.mp4 pred= B correct= B is_correct= True raw= 'B'
Processing level_1_sample_001_visual_boundary_original
level_1_sample_001_visual_boundary.mp4 pred= A correct= A is_correct= True raw= 'A'
Processing level_1_sample_001_visual_boundary_swapped
level_1_sample_001_visual_boundary.mp4 pred= B correct= B is_correct= True raw= 'B'
Processing level_1_sample_001_audio_bo

## Analyze Ladder Results

This aggregates all Qwen3 ladder runs and computes the 6-level accuracy plot, paired boundary comparisons, and swap-consistency diagnostics.



In [18]:
ANALYSIS_DIR = f"analysis/{DATASET_VERSION}_ladder"

!python scripts/analyze_results.py \
  --input "$RESULT_DIR" \
  --dataset_name_prefix "ladder_v2_level_" \
  --output_dir "$ANALYSIS_DIR" \
  --plots

Analyzed 1440 rows from 7 raw result file(s).
Saved analysis to analysis/ladder_v2_ladder


## Inspect Saved Files


In [19]:
!find "$RESULT_DIR" -maxdepth 4 -type f | sort | tail -60
!find analysis -maxdepth 2 -type f | sort


/content/vlm-event-boundary/results/qwen2vl_2b/ladder_v1/.gitkeep
/content/vlm-event-boundary/results/qwen3vl/ladder_v1/.gitkeep
/content/vlm-event-boundary/results/Qwen_Qwen3-VL-8B-Instruct/ladder_v2_level_1_simple/20260601_130050/config.json
/content/vlm-event-boundary/results/Qwen_Qwen3-VL-8B-Instruct/ladder_v2_level_1_simple/20260601_130050/raw_results.jsonl
/content/vlm-event-boundary/results/Qwen_Qwen3-VL-8B-Instruct/ladder_v2_level_1_simple/20260601_130050/summary.json
/content/vlm-event-boundary/results/Qwen_Qwen3-VL-8B-Instruct/ladder_v2_level_2_randomized/20260601_130359/config.json
/content/vlm-event-boundary/results/Qwen_Qwen3-VL-8B-Instruct/ladder_v2_level_2_randomized/20260601_130359/raw_results.jsonl
/content/vlm-event-boundary/results/Qwen_Qwen3-VL-8B-Instruct/ladder_v2_level_2_randomized/20260601_130359/summary.json
/content/vlm-event-boundary/results/Qwen_Qwen3-VL-8B-Instruct/ladder_v2_level_3_non_target_static_distractors/20260601_130734/config.json
/content/vlm-even